# Day 7 (Bonus) — LangChain Edition of the RAG Capstone

---

Days 1–6 taught you RAG **from raw components** — `pypdf`, `sentence-transformers`, `chromadb`, `together`, `httpx`. That's the fastest way to *understand* how RAG works.

But **most job postings mention "LangChain"** as a required skill. Today: rebuild the exact same capstone from Day 6, using LangChain primitives, so you can pattern-match the mapping.

Same endpoints. Same behavior. Same guardrails. Different vocabulary.

**By the end of 75 minutes you'll be able to:**
1. Recognize the LangChain equivalent for every step of a RAG pipeline
2. Chain them together with **LCEL** (LangChain Expression Language — the `|` pipe syntax)
3. Read and modify LangChain-heavy codebases at work
4. Explain the tradeoffs of "raw" vs "LangChain" in an interview


## 1. The mapping table — memorize this

| What it does | Raw code (Days 1–6) | LangChain equivalent |
|---|---|---|
| Load a PDF | `pypdf.PdfReader(path).pages` | `PyPDFLoader(path).load()` |
| Load a DOCX | `docx.Document(path)` | `Docx2txtLoader(path).load()` |
| Load a web page | `trafilatura.fetch_url(url)` | `WebBaseLoader(url).load()` |
| Chunk text | custom `recursive_chunk()` | `RecursiveCharacterTextSplitter().split_documents(docs)` |
| Embed | `SentenceTransformer(...).encode(texts)` | `HuggingFaceEmbeddings(model_name=...)` |
| Store & search | `chromadb.PersistentClient(...).query(...)` | `Chroma(embedding_function=...)` + `.as_retriever(...)` |
| Rerank | Manual `CrossEncoder.predict` | `ContextualCompressionRetriever` + `CrossEncoderReranker` |
| Prompt template | f-string in Python | `ChatPromptTemplate.from_messages(...)` |
| Call the LLM | `together.chat.completions.create` | `ChatTogether(...)` |
| Chain the steps | manual function calls | LCEL: `prompt | llm | StrOutputParser()` |
| Stream tokens | manual iteration on stream | `chain.stream(...)` |

**The trick:** every LangChain object is a **Runnable** with `.invoke()` / `.stream()` / `.batch()` methods. Chain them with `|`. That's LCEL.


## 2. Setup


In [ ]:
!pip install -q \
    langchain>=0.3 langchain-core>=0.3 \
    langchain-community>=0.3 \
    langchain-chroma>=0.2 \
    langchain-together>=0.2 \
    langchain-huggingface>=0.2 \
    langchain-text-splitters>=0.3 \
    pypdf docx2txt beautifulsoup4 \
    python-dotenv


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("TOGETHER_API_KEY"), "Set TOGETHER_API_KEY in .env"


## 3. Loaders — every format becomes a `Document`

Each LangChain loader returns a list of `Document` objects, each with a `.page_content` (the text) and `.metadata` (a dict). Universal shape.


In [ ]:
from langchain_community.document_loaders import (
    PyPDFLoader, Docx2txtLoader, WebBaseLoader, TextLoader,
)


def load(path_or_url: str):
    if path_or_url.startswith("http"):
        return WebBaseLoader(path_or_url).load()
    if path_or_url.endswith(".pdf"):
        return PyPDFLoader(path_or_url).load()
    if path_or_url.endswith(".docx"):
        return Docx2txtLoader(path_or_url).load()
    return TextLoader(path_or_url, encoding="utf-8").load()


# docs = load("some_file.pdf")
# print(len(docs), "documents")
# print(docs[0].metadata)          # {'source': '...', 'page': 0}
# print(docs[0].page_content[:200])


**Compare to Day 2 (raw):** we had a `LoaderRegistry` dict + custom functions returning strings. LangChain does the same dispatch pattern internally, and every loader hands back a uniform `Document` shape — no per-format string-cleanup afterwards.

For unknown formats, `UnstructuredFileLoader` handles ~30 types (HTML, EPUB, PPTX, images with OCR, ...). Requires `unstructured` + system deps; skip until you need it.


## 4. Chunking — `RecursiveCharacterTextSplitter`

The **exact same algorithm** we hand-wrote on Day 2 Section 5, just battle-tested and configurable.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    # Order matters: try each separator, only fall to the next when a chunk is too big
    separators=["\n\n", "\n", ". ", " ", ""],
)

# chunks = splitter.split_documents(docs)
# print(len(chunks), "chunks. First chunk metadata:", chunks[0].metadata)


**What you get for free** vs the custom `recursive_chunk`:

- Metadata is **preserved** on each chunk (source file, page number, etc.). No need to plumb it yourself.
- The separator list lets you tune for prose vs code vs markdown (`MarkdownHeaderTextSplitter` also exists).
- `TokenTextSplitter` when you want to split on tokens instead of chars.


## 5. Embeddings + vector store — one line each


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# vs. raw: chromadb.PersistentClient(...).get_or_create_collection("kb")
vstore = Chroma(
    collection_name="langchain_kb",
    embedding_function=embeddings,
    persist_directory="./langchain_db",
)

# Add documents - LangChain calls .encode() on your behalf
# vstore.add_documents(chunks)


**Retriever** — the abstraction on top of any vector store. Every LangChain retriever exposes `.invoke(query)` returning a list of `Document`s.


In [ ]:
# k=20 candidates before reranking (matches Day 6's CANDIDATE_POOL)
retriever = vstore.as_retriever(search_kwargs={"k": 20})
# hits = retriever.invoke("What is the refund policy?")
# for d in hits[:3]: print(d.page_content[:80])


## 6. Reranking — `ContextualCompressionRetriever`

On Day 3 we hand-rolled `retrieve_and_rerank(question)`. In LangChain, you **wrap** the base retriever with a `ContextualCompressionRetriever` and a `CrossEncoderReranker`. The wrapper transparently reranks every call.


In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

reranker_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
compressor = CrossEncoderReranker(model=reranker_model, top_n=5)

reranked = ContextualCompressionRetriever(
    base_retriever=retriever,       # k=20 wide net
    base_compressor=compressor,     # narrows to top_n=5
)

# hits = reranked.invoke("What is the refund policy?")
# for d in hits: print(d.page_content[:80])


**Compare to Day 3 (raw)**: 15 lines of `retrieve_and_rerank`. Here: 3 lines. The `ContextualCompressionRetriever` name is unwieldy but the API is clean — it's just another `Retriever` you can `.invoke()`.


## 7. Prompt — `ChatPromptTemplate`

The exact same numbered-context + citation template from Day 4, just written declaratively.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM = (
    "You are a helpful assistant. Answer using ONLY the numbered context. "
    "If the answer is not there, say 'I don't know.' "
    "Cite sources with bracket numbers.\n\n"
    "Format your answer like this:\n"
    "The Pro plan costs $29/month [2]."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("user", "Context:\n{context}\n\nQuestion: {question}"),
])

# `context` and `question` are template variables filled at .invoke() time
# print(prompt.invoke({"context": "[1] Pro plan is $29/mo", "question": "How much is Pro?"}))


## 8. LLM — `ChatTogether`

LangChain wraps Together AI in a `ChatModel` object with the standard `.invoke()` / `.stream()` interface. Same OSS default (`openai/gpt-oss-20b`) as the rest of the course.


In [ ]:
from langchain_together import ChatTogether

llm = ChatTogether(model="openai/gpt-oss-20b", temperature=0.2)

# r = llm.invoke([("user", "Say hi in five words.")])
# print(r.content)


## 9. Chain the steps with LCEL — `|` is the whole language

LCEL (LangChain Expression Language) is *just Python's `|` operator*. Every LangChain object is a **Runnable**; `|` means "pipe the output of the left into the right".

Our RAG chain, spelled out:

```
retrieved_docs -> format them as a numbered string -> stuff into prompt -> llm -> parse to plain string
```

In LCEL:


In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


def format_docs(docs) -> str:
    return "\n\n".join(f"[{i+1}] {d.page_content}" for i, d in enumerate(docs))


rag_chain = (
    {
        "context":  reranked | format_docs,     # retrieve+rerank, then format
        "question": RunnablePassthrough(),      # pass the raw question through
    }
    | prompt
    | llm
    | StrOutputParser()
)

# answer = rag_chain.invoke("What is the refund policy?")
# print(answer)


**How to read that chain:**

- The `{...}` dict at the start says: build two named inputs for the prompt — `context` (from the reranked retriever, then formatted) and `question` (passed through as-is).
- `prompt` turns that dict into a formatted `ChatPromptValue`.
- `llm` runs it and returns an AI message.
- `StrOutputParser()` extracts the string content.

Every arrow is `.invoke()` under the hood. Try `rag_chain.batch([...])` to fan out multiple queries, or `rag_chain.astream(...)` for async streaming — same chain, different verbs.


## 10. Streaming with `.stream()`

Day 4/6 hand-wrote a token-yielding generator. In LangChain, any chain that ends in a chat model gets `.stream()` for free.


In [ ]:
# for chunk in rag_chain.stream("What is the refund policy?"):
#     print(chunk, end="", flush=True)
# print()


Under the hood, the streaming reaches into the `ChatTogether` step and yields tokens as they arrive. The upstream retriever + rerank still runs synchronously (they're not streamable), then the LLM streams. Feels the same to the caller.


## 11. The full app — see `main.py`

`main.py` in this folder is the **full FastAPI rewrite** of Day 6's capstone using LangChain. Same endpoints:

- `POST /login` → JWT
- `POST /ingest` → LangChain loaders + splitter + Chroma
- `POST /ask` → streaming LCEL chain with injection filter + refusal
- `GET /usage` → per-user token + cost log

Compare it side-by-side with Day 6's `main.py`:

```bash
diff -u ../Day_6_Capstone_RAG_Chatbot/main.py main.py
```

Total lines are similar. The LangChain version has ~30% less business logic (chunking, retrieval, prompting) and adds ~10 lines of framework imports. **Net win when the pipeline gets more complex** — for our capstone the win is small.


## 12. Tradeoffs — when to reach for LangChain

**Reach for LangChain when:**
- Your team already uses it (matching an existing codebase is worth 10× the marginal ergonomics)
- You need built-ins we didn't cover: `MultiQueryRetriever`, `SelfQueryRetriever`, `EnsembleRetriever`, `ParentDocumentRetriever`, dozens of loaders, LangSmith tracing
- The pipeline has **many steps** and you want them composable + swappable

**Skip it (write raw) when:**
- You're learning — see Days 1–6 for why
- The pipeline is small and you value 100% control
- You care about long-term stability more than composability (LangChain has churned versions frequently; raw code is forever)

**LlamaIndex** is the sibling library — very similar concepts, slightly different API, heavier lean toward RAG (vs LangChain's agents lean). Interviews sometimes mention both. If you learn LangChain first, LlamaIndex takes an afternoon to pick up.


## Recap

- Every step of RAG has a **direct LangChain equivalent**. The mapping table in §1 is the whole thing.
- **LCEL (`|` pipe)** chains Runnables. That's the syntax; the objects are ordinary Python.
- `ContextualCompressionRetriever + CrossEncoderReranker` collapses 15 lines of Day 3 into 3.
- `.stream()` gives you streaming for free.
- **Raw code from Days 1–6** is still what you should reach for when learning — LangChain is a productivity tool once you already understand what's happening under the hood.
- **See `main.py`** for the full FastAPI rewrite of Day 6's capstone.

This is the last day of Section 6. Next: **Section 7 — AI Agents.**
